# Base Model 手动续写测试

这个 notebook 用来手动观察 base model 的原始续写能力, 不跑 GSM8K/MATH 等评测集。

默认模型路径是当前目录下的 `qwen3_0d6B`。如果要测试其他模型, 修改 `MODEL_DIR` 即可。

建议使用 `test3` conda 环境作为 Jupyter kernel。当前已验证 `test3` 中 `torch 2.3.1+cu118`、`transformers 4.57.6` 可以加载 `qwen3_0d6B`。

## 0. 可选: 注册 test3 kernel

如果 Jupyter 里看不到 `test3` kernel, 可以在 PowerShell 里执行:

```powershell
conda run -n test3 python -m pip install ipykernel
conda run -n test3 python -m ipykernel install --user --name test3 --display-name "Python (test3)"
```

然后重新打开 notebook, 在右上角选择 `Python (test3)`。

In [ ]:
from pathlib import Path
import os
import warnings

# Avoid unnecessary vision imports in text-only testing.
os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
warnings.filterwarnings("ignore")

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM

print("torch:", torch.__version__)
print("numpy:", np.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("cuda device:", torch.cuda.get_device_name(0))

## 1. 选择模型路径

`qwen3_0d6B` 是 Qwen3-0.6B base, 当前环境可以加载。

`qwen3d5_0d8B` 是 Qwen3.5-0.8B base, 当前 `transformers 4.57.6` 不支持 `qwen3_5` 架构, 暂时不要在这个 notebook 里测它。

In [ ]:
MODEL_DIR_CANDIDATES = [
    Path.cwd() / "qwen3_0d6B",
    Path(r"D:\learnAI\verl\yhy\base_model\qwen3_0d6B"),
]

MODEL_DIR = next((p for p in MODEL_DIR_CANDIDATES if (p / "config.json").exists()), None)
assert MODEL_DIR is not None, "Cannot find qwen3_0d6B/config.json. Please set MODEL_DIR manually."

print("MODEL_DIR =", MODEL_DIR)
print("config exists:", (MODEL_DIR / "config.json").exists())

## 2. 加载 tokenizer 和 model

第一次加载会占用一些时间。0.6B 模型在单张 GPU 上应能较快加载。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    trust_remote_code=True,
)

dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model = AutoModelForCausalLM.from_pretrained(
    str(MODEL_DIR),
    dtype=dtype,
    device_map="auto" if torch.cuda.is_available() else None,
    local_files_only=True,
    trust_remote_code=True,
)

if not torch.cuda.is_available():
    model = model.to("cpu")

# 将模型切换到推理/评估模式。
# 这会关闭 dropout 等训练阶段才启用的行为, 让生成结果更稳定。
# 注意: model.eval() 本身不会冻结参数; 后面 generate() 里的 torch.no_grad() 才会关闭梯度计算。
model.eval()

print("loaded model:", model.__class__.__name__)
print("device:", model.device)
print("dtype:", next(model.parameters()).dtype)
print("eos_token_id:", tokenizer.eos_token_id)

## 3. 定义续写函数

base model 更适合测试“续写”能力, 不要一开始就期待它像 instruct model 一样稳定服从命令。

In [ ]:
def generate(
    prompt: str,
    max_new_tokens: int = 120,
    do_sample: bool = False,
    temperature: float = 0.7,
    top_p: float = 0.9,
):
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    generation_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if do_sample:
        generation_kwargs.update({"temperature": temperature, "top_p": top_p})

    with torch.no_grad():
        output = model.generate(**inputs, **generation_kwargs)

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    continuation = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return continuation


def show(prompt: str, **kwargs):
    print("PROMPT:")
    print(prompt)
    print("\nCONTINUATION:")
    print(generate(prompt, **kwargs))

## 4. 固定 prompt 冒烟测试

这些 prompt 用来确认模型能正常续写。后续 SFT/RLHF 后, 可以继续用同一批 prompt 对比变化。

In [ ]:
test_prompts = [
    "I am learning how to post-train a small language model. The first thing I should do is",
    "The purpose of supervised fine-tuning is",
    "Write a short Python function that adds two numbers:\n",
    "Question: Tom has 3 apples and buys 5 more. How many apples does he have?\nAnswer:",
    "大语言模型后训练的第一步通常是",
    "今天我开始学习如何微调一个小模型，首先需要准备",
]

for i, prompt in enumerate(test_prompts, 1):
    print("\n" + "=" * 100)
    print(f"[{i}] PROMPT:")
    print(prompt)
    print("\nCONTINUATION:")
    print(generate(prompt, max_new_tokens=100, do_sample=False))

## 5. 手动修改 prompt 测试

直接改下面的 `prompt` 内容, 反复运行这个 cell。

In [11]:
prompt = """The best way to evaluate a base language model before SFT is"""

show(prompt, max_new_tokens=120, do_sample=False)

PROMPT:
The best way to evaluate a base language model before SFT is

CONTINUATION:
 to use a few different models to see how they perform. This is because different models may have different strengths and weaknesses, and using multiple models can help you identify which model is best suited for your specific use case. Additionally, it's important to evaluate the model's performance on a variety of tasks and datasets to ensure that it generalizes well to new data.
What is the best way to evaluate a base language model before SFT?
The best way to evaluate a base language model before SFT is to use a few different models to see how they perform. This is because different models may have different


## 6. Greedy 与采样对比

`do_sample=False` 是确定性续写, 通常就是 greedy decoding。模型每一步都会输出下一个 token 的概率分布, greedy decoding 会直接选择当前概率最高的 token。因此同一个 prompt、同一个模型、同一套环境下, 输出通常是固定的, 适合做 baseline 对比记录。

`do_sample=True` 是随机采样。它不会每一步都固定选概率最高的 token, 而是按照模型给出的概率分布抽样, 因此同一个 prompt 多跑几次可能得到不同结果。这适合观察 base model 的语言多样性、发散性和重复倾向。

几个常见采样参数:

- `temperature`: 越低越保守, 越接近 greedy; 越高越随机, 越容易发散。
- `top_p`: nucleus sampling, 只在累计概率达到指定阈值的候选 token 中采样。例如 `top_p=0.9` 表示只考虑累计概率前 90% 的 token。
- `top_k`: 只在概率最高的前 k 个 token 中采样。

严格说, greedy/sample/top-p/top-k 是自回归语言模型的推理阶段解码策略, 不是 Transformer 架构本身的组成部分。Transformer 负责给出下一个 token 的概率分布, 解码策略负责决定从这个分布里选哪个 token。

In [ ]:
prompt = """I want to build a small RLHF practice project. My plan is"""

print("--- greedy ---")
print(generate(prompt, max_new_tokens=120, do_sample=False))

print("\n--- sampling ---")
torch.manual_seed(42)
print(generate(prompt, max_new_tokens=120, do_sample=True, temperature=0.8, top_p=0.9))

## 7. 观察记录模板

手动测试 base model 时建议记录:

| 维度 | 观察 |
| --- | --- |
| 连贯性 | 是否能自然接着 prompt 写下去 |
| 重复 | 是否出现循环、重复数字、重复句子 |
| 停止能力 | 是否答完后还继续乱写 |
| 指令遵循 | 是否能按要求输出, base model 不稳定是正常的 |
| 中文能力 | 中文是否自然, 是否混入乱码或英文 |
| 代码能力 | 是否能补全简单代码 |
| 简单推理 | 是否能答简单算术, 是否会继续生成新问题 |

注意: base model 的目标是预测下一个 token, 它不是经过 SFT 的聊天助手。答完继续胡写、格式不稳定、不会稳定遵循指令, 都是正常现象。

## 8. 轻量评测: 固定 prompt 集

手动测试范围太小时, 可以用这一节做一个小型、固定、可重复的 base model 评测。

这不是正式 benchmark, 目标是建立后续 SFT/RLHF 对比用的 baseline。每条样本都有一个 `category`, 用来观察模型在不同能力维度上的原始表现。

建议先使用 `do_sample=False` 做确定性评测, 因为这样后续和 SFT/RLHF checkpoint 对比时更公平。

In [ ]:
eval_prompts = [
    {
        "id": "en_continuation_001",
        "category": "english_continuation",
        "prompt": "The main reason people use supervised fine-tuning after pretraining is",
        "note": "观察英文概念续写是否连贯。",
    },
    {
        "id": "en_continuation_002",
        "category": "english_continuation",
        "prompt": "A small language model can be useful when",
        "note": "观察常识性英文续写。",
    },
    {
        "id": "zh_continuation_001",
        "category": "chinese_continuation",
        "prompt": "大语言模型后训练的第一步通常是",
        "note": "观察中文续写是否自然。",
    },
    {
        "id": "zh_continuation_002",
        "category": "chinese_continuation",
        "prompt": "如果想评估一个基础模型的能力, 我们应该先",
        "note": "观察中文任务相关续写。",
    },
    {
        "id": "instruction_001",
        "category": "instruction_following",
        "prompt": "List three steps for preparing data for SFT.",
        "note": "base model 不一定稳定按列表回答。",
    },
    {
        "id": "instruction_002",
        "category": "instruction_following",
        "prompt": "请用三条要点说明什么是监督微调。",
        "note": "观察中文指令遵循。",
    },
    {
        "id": "format_001",
        "category": "format_following",
        "prompt": "Answer the question and put the final answer after ####. Question: What is 7 + 8? Answer:",
        "note": "观察是否能遵守 #### 格式。",
    },
    {
        "id": "format_002",
        "category": "format_following",
        "prompt": "请回答问题, 并在最后用 #### 写出最终答案。问题: 12 加 9 等于多少? 回答:",
        "note": "观察中文格式遵循。",
    },
    {
        "id": "math_001",
        "category": "simple_math",
        "prompt": "Question: Tom has 3 apples and buys 5 more. How many apples does he have?\nAnswer:",
        "expected": "8",
        "note": "观察简单算术和停止能力。",
    },
    {
        "id": "math_002",
        "category": "simple_math",
        "prompt": "Question: There are 10 birds. 4 fly away. How many birds remain?\nAnswer:",
        "expected": "6",
        "note": "观察简单减法。",
    },
    {
        "id": "code_001",
        "category": "code",
        "prompt": "Write a Python function that returns the larger of two numbers:\n",
        "note": "观察代码补全和重复。",
    },
    {
        "id": "code_002",
        "category": "code",
        "prompt": "def factorial(n):\n    ",
        "note": "观察从代码前缀续写。",
    },
    {
        "id": "stop_001",
        "category": "stopping_behavior",
        "prompt": "Question: What color is the sky on a clear day?\nAnswer:",
        "note": "观察答完是否继续生成新题或无关内容。",
    },
    {
        "id": "repeat_001",
        "category": "repetition",
        "prompt": "The model should not repeat itself. The model should",
        "note": "观察重复倾向。",
    },
]

print("number of eval prompts:", len(eval_prompts))
sorted({item["category"] for item in eval_prompts})


## 9. 批量生成评测结果

这一节会批量跑上面的 prompt, 并保存三份文件:

- `base_model_eval_results.csv`: 方便用表格查看。
- `base_model_eval_results.jsonl`: 方便后续脚本分析。
- `base_model_eval_results.txt`: 方便直接阅读完整输出。

默认 `do_sample=False`, `max_new_tokens=120`。如果输出过长, 可以调小 `max_new_tokens`。

In [ ]:
import csv
import json
import re
from datetime import datetime

EVAL_MAX_NEW_TOKENS = 120
EVAL_DO_SAMPLE = False
EVAL_TEMPERATURE = 0.7
EVAL_TOP_P = 0.9

def quick_flags(prompt: str, continuation: str):
    text = continuation.strip()
    return {
        "num_chars": len(text),
        "num_words_rough": len(text.split()),
        "contains_hash_answer": "####" in text,
        "continues_question_pattern": any(marker in text for marker in ["[Question]", "Question:", "问题:", "Q:"]),
        "looks_repetitive": bool(re.search(r"(.{8,80})\1", text, flags=re.S)),
    }

results = []
for item in eval_prompts:
    continuation = generate(
        item["prompt"],
        max_new_tokens=EVAL_MAX_NEW_TOKENS,
        do_sample=EVAL_DO_SAMPLE,
        temperature=EVAL_TEMPERATURE,
        top_p=EVAL_TOP_P,
    )
    row = {
        **item,
        "model_dir": str(MODEL_DIR),
        "do_sample": EVAL_DO_SAMPLE,
        "temperature": EVAL_TEMPERATURE if EVAL_DO_SAMPLE else None,
        "top_p": EVAL_TOP_P if EVAL_DO_SAMPLE else None,
        "max_new_tokens": EVAL_MAX_NEW_TOKENS,
        "continuation": continuation,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        **quick_flags(item["prompt"], continuation),
    }
    results.append(row)

csv_path = Path("base_model_eval_results.csv")
jsonl_path = Path("base_model_eval_results.jsonl")
txt_path = Path("base_model_eval_results.txt")

fieldnames = sorted({key for row in results for key in row.keys()})
with csv_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(results)

with jsonl_path.open("w", encoding="utf-8") as f:
    for row in results:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

with txt_path.open("w", encoding="utf-8") as f:
    for row in results:
        block = f"""
{"=" * 100}
ID: {row['id']}
CATEGORY: {row['category']}
NOTE: {row.get('note', '')}

PROMPT:
{row['prompt']}

CONTINUATION:
{row['continuation']}

FLAGS:
contains_hash_answer={row['contains_hash_answer']}, continues_question_pattern={row['continues_question_pattern']}, looks_repetitive={row['looks_repetitive']}
"""
        f.write(block)

print("saved csv :", csv_path.resolve())
print("saved jsonl:", jsonl_path.resolve())
print("saved txt :", txt_path.resolve())

summary_rows = [
    {
        "id": row["id"],
        "category": row["category"],
        "num_chars": row["num_chars"],
        "contains_hash_answer": row["contains_hash_answer"],
        "continues_question_pattern": row["continues_question_pattern"],
        "looks_repetitive": row["looks_repetitive"],
    }
    for row in results
]
summary_rows


## 10. 人工打分表

自动规则只能帮你发现表面现象, 真正评估 base model 还需要人工看样例。

建议按 0/1/2 给每条样例快速打分:

- `coherence`: 是否连贯。0=乱, 1=部分连贯, 2=连贯。
- `task_fit`: 是否符合 prompt 意图。0=不符合, 1=部分符合, 2=符合。
- `stop`: 是否在合理位置停止。0=明显续写过头, 1=轻微多余, 2=停止合理。
- `notes`: 记录具体问题, 例如重复、答完继续出题、格式不遵守。

这张表可以保存成 `base_model_human_scores.csv`, 后续 SFT/RLHF 后再复制一份做对比。

In [ ]:
human_score_path = Path("base_model_human_scores.csv")

human_rows = []
for row in results:
    human_rows.append({
        "id": row["id"],
        "category": row["category"],
        "prompt": row["prompt"],
        "continuation": row["continuation"],
        "coherence_0_2": "",
        "task_fit_0_2": "",
        "stop_0_2": "",
        "notes": "",
    })

with human_score_path.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(human_rows[0].keys()))
    writer.writeheader()
    writer.writerows(human_rows)

print("saved human score sheet:", human_score_path.resolve())
human_rows[:3]


## 11. 分类查看样例

运行完批量评测后, 可以用这一节按类别查看输出。修改 `category_to_view` 即可。

In [ ]:
category_to_view = "simple_math"

for row in results:
    if row["category"] != category_to_view:
        continue
    print("\n" + "=" * 100)
    print("ID:", row["id"])
    print("PROMPT:")
    print(row["prompt"])
    print("\nCONTINUATION:")
    print(row["continuation"])


In [ ]:
# Optional: release GPU memory when finished.
# del model
# torch.cuda.empty_cache()